# Lab 1 - Mock

**Ejercicio 1: Aritmética de Precisión Finita**

Dada la función $f(x) = \frac{1-\cos(x)}{x^2}$:

1. Implementa una función en Python que evalúe $f(x)$ para $x = 10^{-5}, 10^{-6}, \dots, 10^{-11}$. Imprime los resultados.
2. Explica en un comentario o `print` qué fenómeno numérico estás observando a medida que $x$ se acerca a 0 y por qué ocurre.
3. Propón una reformulación algebraica equivalente de $f(x)$ que solucione este problema, impleméntala como `f_mejorada(x)` y demuestra que ahora los valores convergen correctamente.

In [1]:
import math

# 1. Definición de la función original
def f(x):
    # ATENCIÓN: Esta forma sufre de "cancelación catastrófica"
    # cuando x es muy pequeño, cos(x) es casi 1. Al restar 1 - casi_1,
    # se pierden los dígitos significativos.
    return (1 - math.cos(x)) / (x**2)

# 2. Reformulación algebraica equivalente (Identidad del ángulo mitad)
# Sabemos que 1 - cos(x) = 2 * sin^2(x/2)
def f_mejorada(x):
    # Esta forma evita restar números casi iguales, preservando la precisión.
    return 2 * (math.sin(x / 2)**2) / (x**2)

print("Evaluación de f(x) vs f_mejorada(x):")
print("-" * 65)
print(f"{'x':<10} | {'f(x) (Inestable)':<25} | {'f_mejorada(x) (Estable)':<25}")
print("-" * 65)

# Bucle sobre las potencias de x = 10^-5 hasta 10^-11
for i in range(5, 12):
    x = 10**(-i)
    val_f = f(x)
    val_f_mejorada = f_mejorada(x)
    print(f"10^-{i:<2}   | {val_f:<25} | {val_f_mejorada:<25}")

# EXPLICACIÓN TEÓRICA PARA EL EXAMEN:
# El límite matemático de f(x) cuando x -> 0 es 0.5 (usando L'Hôpital).
# En precisión finita (ordenador), cos(x) para x minúsculo se redondea a 1.0.
# Entonces 1.0 - 1.0 = 0.0, y al dividir entre x^2 da error o 0.0.
# A este fenómeno se le llama "Cancelación Catastrófica por resta de números cercanos".
# Con la reformulación no restamos nada que se cancele, por lo que converge bien al 0.5.


Evaluación de f(x) vs f_mejorada(x):
-----------------------------------------------------------------
x          | f(x) (Inestable)          | f_mejorada(x) (Estable)  
-----------------------------------------------------------------
10^-5    | 0.5000000413701854        | 0.4999999999958333       
10^-6    | 0.5000444502911705        | 0.4999999999999583       
10^-7    | 0.4996003610813205        | 0.4999999999999996       
10^-8    | 0.0                       | 0.5                      
10^-9    | 0.0                       | 0.5                      
10^-10   | 0.0                       | 0.5                      
10^-11   | 0.0                       | 0.5                      


**Ejercicio 2: Bisección y Control de Errores**

Dada la ecuación $f(x) = \ln(x) + x = 0$:

1. Sabiendo que hay una raíz en el intervalo $[0.1, 1.0]$, calcula matemáticamente (puedes programarlo o poner el resultado en un comentario) el número mínimo de iteraciones $N$ necesarias para asegurar un error absoluto menor o igual a $\epsilon = 10^{-4}$.
2. Implementa desde cero la función `biseccion(f, a, b, tol)` y utilízala para encontrar la raíz en dicho intervalo con la tolerancia especificada. El programa debe imprimir la raíz aproximada y el número de iteraciones reales que le ha tomado.


In [2]:
import math  # noqa: F811

# 1. CÁLCULO TEÓRICO: Número de iteraciones
a = 0.1
b = 1.0
tol = 1e-4

# Fórmula del error absoluto en Bisección: err_ab = (b - a) / (2^N)
# Exigiendo err_ab <= tol => N >= log2((b-a)/tol)
N_min = math.ceil(math.log2((b - a) / tol))

print("1. Cálculo Matemático del número mínimo de iteraciones N:")
print(f"N >= log2(({b} - {a}) / {tol})")
print(f"N >= log2({(b - a)/tol}) ≈ {math.log2((b-a)/tol):.4f}")
print(f"Por tanto, N mínimo absoluto necesario: {N_min} iteraciones.\n")


# 2. DEFINICIÓN DEL ALGORITMO: Bisección
def biseccion(f, a, b, tol):
    """
    f: Función objetivo
    a, b: Límite inferior y superior del intervalo de búsqueda
    tol: Tolerancia (criterio de parada del tamaño del intervalo)
    """
    # 3. COMPROBACIÓN TEÓRICA INICIAL: Teorema de Bolzano.
    # Si la función no cruza el 0, no hay raíz asegurada en [a,b]
    if f(a) * f(b) > 0:
        raise ValueError("f(a) y f(b) tienen el mismo signo. No se puede asegurar raíz.")

    iter_count = 0

    # 4. BUCLE DE ITERACIÓN
    # La cota de error es (b - a)/2. Paramos cuando esto es menor que la tolerancia.
    while (b - a) / 2.0 > tol:
        # Calcular el punto medio (bisección del intervalo)
        c = (a + b) / 2.0
        iter_count += 1

        # Evaluar la función en c
        f_c = f(c)

        # CRITERIO EXACTO: Si llegamos exactamente a 0
        if f_c == 0:
            return c, iter_count

        # TEOREMA DEL VALOR MEDIO/BOLZANO (Cambio de signo)
        if f(a) * f_c < 0:
            # Hay cambio de signo de a .. c, por lo que la raíz está en el intervalo [a, c]
            b = c  # Bajar el límite superior
        else:
            # El cambio de signo está en el intervalo [c, b]
            a = c  # Subir el límite inferior

    # La raíz aproximada al final es el último punto medio
    raiz_aprox = (a + b) / 2.0
    return raiz_aprox, iter_count

# 5. USO DEL ALGORITMO
def f_ej2(x):
    return math.log(x) + x

raiz, iter_reales = biseccion(f_ej2, a, b, tol)

print("2. Resultados del algoritmo de Bisección:")
print(f"Raíz aproximada log(x)+x=0: {raiz:.6f}")
print(f"Iteraciones reales tomadas: {iter_reales}")
print(f"Evaluación de f(x) en la raíz: {f_ej2(raiz):.6e}")


1. Cálculo Matemático del número mínimo de iteraciones N:
N >= log2((1.0 - 0.1) / 0.0001)
N >= log2(9000.0) ≈ 13.1357
Por tanto, N mínimo absoluto necesario: 14 iteraciones.

2. Resultados del algoritmo de Bisección:
Raíz aproximada log(x)+x=0: 0.567194
Iteraciones reales tomadas: 13
Evaluación de f(x) en la raíz: 1.390224e-04


**Ejercicio 3: Punto Fijo y Transformaciones**

Dada la ecuación no lineal $f(x) = x^2 - x - 1 = 0$:

1. Implementa la función `punto_fijo(g, x0, tol, maxit)`.
2. Transforma la ecuación en la forma $x = g(x)$ utilizando la expresión $g(x) = 1 + \frac{1}{x}$.
3. Utiliza tu función con $x_0 = 1.0$, una tolerancia de $10^{-4}$ y un máximo de 50 iteraciones para encontrar la raíz.
4. En un comentario, explica brevemente cómo se vería el proceso de convergencia de esta iteración concreta si lo dibujáramos gráficamente (su interpretación geométrica).

In [3]:
# 1. IMPLEMENTACIÓN DE PUNTO FIJO
def punto_fijo(g, x0, tol, maxit):
    """
    g: Función de iteración (reorganización de f(x)=0 a x=g(x))
    x0: Semilla inicial o valor de partida
    tol: Tolerancia requerida (parada cuando |x_i - x_{i-1}| < tol)
    maxit: Máximo de iteraciones (seguridad por si diverge)
    """
    print("Iniciando algoritmo de Punto Fijo...")
    print("-" * 50)
    print(f"{'Iter':<5} | {'x_n':<15} | {'|x_n - x_n-1|':<15}")
    print("-" * 50)

    # Evaluar iterativamente la nueva aproximación: x_{i} = g(x_{i-1})
    x_prev = x0
    for i in range(1, maxit + 1):
        x_new = g(x_prev)

        # Calcular el residuo o error absoluto respecto a la iteración anterior
        error_abs = abs(x_new - x_prev)

        # Mostrar traza
        print(f"{i:<5} | {x_new:<15.6f} | {error_abs:<15.6e}")

        # CRITERIO DE PARADA (Condición de convergencia)
        if error_abs < tol:
            print("-" * 50)
            print(f"-> ¡Convergencia lograda en la iteración {i}!")
            return x_new, i

        # Actualizar memoria para siguiente vuelta
        x_prev = x_new

    print("-" * 50)
    print(f"-> ¡Alerta! Máximo número de iteraciones ({maxit}) alcanzado sin converger.")
    return x_new, maxit

# 2. TRANSFORMACIÓN: g(x) = 1 + 1/x
# Ojo, no dividir por cero. Asumimos x0 > 0.
def g_ej3(x):
    return 1.0 + (1.0 / x)

# 3. EJECUCIÓN
x0 = 1.0
tol_pf = 1e-4
maxit = 50

raiz_pf, iter_pf = punto_fijo(g_ej3, x0, tol_pf, maxit)

# 4. INTERPRETACIÓN GEOMÉTRICA (COBWEB / TELARAÑA)
# Dado g(x) = 1 + 1/x, su derivada es g'(x) = -1/x^2
# Al ser g'(1.618) ≈ -0.38 < 0, es un valor negativo pero menor de 1 en valor absoluto.
# La interpretación gráfica (Cobweb) muestra que en cada evaluación pasamos la
# función a la diagonal y=x, resultando en que la "telaraña" hace una ESPIRAL
# HASTA CONVERGER hacia adentro cruzando repetidamente el punto fijo.
# Si el g'(x)>0 (positivo) el dibujo iría aproximándose lentamente (o en escalera) de un lado.
print("\nRaíz encontrada con Punto Fijo:", round(raiz_pf, 6))
print("Nota geométrica: La convergencia es en ESPIRAL al ser g'(x) negativo (-1/x^2).")


Iniciando algoritmo de Punto Fijo...
--------------------------------------------------
Iter  | x_n             | |x_n - x_n-1|  
--------------------------------------------------
1     | 2.000000        | 1.000000e+00   
2     | 1.500000        | 5.000000e-01   
3     | 1.666667        | 1.666667e-01   
4     | 1.600000        | 6.666667e-02   
5     | 1.625000        | 2.500000e-02   
6     | 1.615385        | 9.615385e-03   
7     | 1.619048        | 3.663004e-03   
8     | 1.617647        | 1.400560e-03   
9     | 1.618182        | 5.347594e-04   
10    | 1.617978        | 2.042901e-04   
11    | 1.618056        | 7.802747e-05   
--------------------------------------------------
-> ¡Convergencia lograda en la iteración 11!

Raíz encontrada con Punto Fijo: 1.618056
Nota geométrica: La convergencia es en ESPIRAL al ser g'(x) negativo (-1/x^2).


**Ejercicio 4: Cálculo del Épsilon de la Máquina**

1. Implementa en Python el algoritmo iterativo para aproximar el épsilon de la máquina.
2. Partiendo de $x = 1.0$, divide $x$ entre $2.0$ repetidamente mientras $1.0 + x \ne 1.0$.
3. Cuando el bucle termine, define y muestra $EPS = 2.0 \cdot x$.
4. Compara tu resultado con el valor nativo de Python usando `numpy.finfo(float).eps`.

In [ ]:
import numpy as np

# EXPLICACIÓN TEÓRICA PARA EL EXAMEN:
# El "épsilon de la máquina" (EPS o U) es el número positivo más pequeño
# tal que 1.0 + EPS > 1.0 en la representación de coma flotante del ordenador.
# Nos da el error relativo de redondeo de la máquina.

# 1. ALGORITMO ITERATIVO DEL ÉPSILON DE LA MÁQUINA
def calcular_epsilon():
    x = 1.0
    iteraciones = 0

    # 2. BUCLE HASTA PERDIDA DE PRECISIÓN (UNDERFLOW de la suma)
    # En un ordenador, debido a la suma finita de mantis, llega
    # un momento en que (1 + num_super_pequeno) da 1.
    while 1.0 + x != 1.0:
        # Se guarda el paso anterior
        x_prñl.ev = x

        # Iteración: dividir el número a la mitad iterativamente
        x /= 2.0

        iteraciones += 1

    print(f"Iteraciones necesarias: {iteraciones}")

    # 3. CÁLCULO DEL ÉPSILON
    # Al finalizar el bucle, x contiene el valor que "desaparece" (último menor que causó que 1.0 + x == 1.0).
    # Entonces el ÚLTIMO valor válido ANTES de cruzar la frontera del 1 (x_prev)
    # multiplicado por 2 (o simplemente 2*x como pide el enunciado) es el verdadero "paso mínimo perceptible".
    # (El número x "murió" en la iteración anterior, por lo tanto EPS = 2.0 * x o simplemente x_prev)

    eps_calculado = 2.0 * x

    return eps_calculado

mi_eps = calcular_epsilon()

# 4. COMPARACIÓN NATIVA CON LA LIBRERÍA NUMPY
eps_numpy = np.finfo(float).eps

print("-" * 50)
print(f"Épsilon calculado manual: {mi_eps}")
print(f"Épsilon de numpy (float64): {eps_numpy}")
print("-" * 50)

# Verificación de coincidencia
if mi_eps == eps_numpy:
    print("¡Éxito! Nuestro cálculo iterativo coincide con la precisión doble nativa.")
else:
    print("El épsilon no coincide, comprueba la arquitectura (32 vs 64 bits).")


Iteraciones necesarias: 53
--------------------------------------------------
Épsilon calculado manual: 2.220446049250313e-16
Épsilon de numpy (float64): 2.220446049250313e-16
--------------------------------------------------
¡Éxito! Nuestro cálculo iterativo coincide con la precisión doble nativa.


**Ejercicio 5: Análisis de Convergencia del Punto Fijo**

Dada la ecuación $f(x) = x^2 - x - 1 = 0$ (cuya raíz positiva es $\phi \approx 1.618$):

1. Transforma la ecuación usando $g_1(x) = x^2 - 1$ y $g_2(x) = \sqrt{x+1}$.
2. Calcula matemáticamente (a mano o en un comentario) la derivada de ambas funciones: $g_1^{\prime}(x)$ y $g_2^{\prime}(x)$.
3. Evalúa el valor absoluto de esas derivadas en la raíz $x \approx 1.618$.
4. Justifica teóricamente cuál de las dos iteraciones convergerá y cuál no, basándote en la condición $|g^{\prime}(x^*)|< 1$.

In [5]:
import math

# 1. Definir ambas funciones g(x) y sus derivadas analíticas g'(x)
# TEOREMA DEL PUNTO FIJO:
# Una iteración x = g(x) CONVERGE a la raíz 'phi' si la derivada de g(x)
# evaluada en la raíz tiene valor absoluto MENOR QUE 1.  |g'(phi)| < 1

# Transforma 1: x^2 - x - 1 = 0  => x = x^2 - 1 = g_1(x)
def g1(x):
    return x**2 - 1

def dg1(x): # 2. Derivada de g1(x) = 2x
    return 2 * x

# Transforma 2: x^2 - x - 1 = 0  => x^2 = x + 1 => x = sqrt(x + 1) = g_2(x)
def g2(x):
    return math.sqrt(x + 1)

def dg2(x): # 2. Derivada de g2(x) = 1 / (2 * sqrt(x + 1))
    return 1.0 / (2.0 * math.sqrt(x + 1))


# Valor de la Raíz Real Phi = Proporción áurea = (1 + sqrt(5))/2
phi = 1.6180339887

# 3. EVALUACIÓN DE LAS DERIVADAS EN LA RAÍZ
val_dg1 = dg1(phi)
val_dg2 = dg2(phi)

print("-" * 50)
print(f"Raíz analizada: phi ≈ {phi:.6f}")
print("-" * 50)
print(f"g1'(x) evaluado en phi: {val_dg1:.6f}")
print(f"|g1'(phi)| = {abs(val_dg1):.6f}")

print("\n")
print(f"g2'(x) evaluado en phi: {val_dg2:.6f}")
print(f"|g2'(phi)| = {abs(val_dg2):.6f}")
print("-" * 50)


# 4. JUSTIFICACIÓN TEÓRICA
print("Análisis de convergencia:")
# Condición para g1
if abs(val_dg1) < 1:
     print(f"-> g1(x) CONVERGIRÁ teóricamente puesto que |g1'(phi)| ({abs(val_dg1):.2f}) < 1")
else:
     print(f"-> g1(x) DIVERGIRÁ teóricamente puesto que |g1'(phi)| ({abs(val_dg1):.2f}) > 1")

# Condición para g2
if abs(val_dg2) < 1:
     print(f"-> g2(x) CONVERGIRÁ teóricamente puesto que |g2'(phi)| ({abs(val_dg2):.2f}) < 1")
else:
     print(f"-> g2(x) DIVERGIRÁ teóricamente puesto que |g2'(phi)| ({abs(val_dg2):.2f}) > 1")

# Nota para examen teórico de convergencia:
# Las iteraciones lineales como un simple "x = F(x)" suelen divergir si no modificas la pendiente
# (por eso existen los factores de relajación "omega" en matrices o métodos super-lineales como Newton que obligan al cero en el g'(phi)).


--------------------------------------------------
Raíz analizada: phi ≈ 1.618034
--------------------------------------------------
g1'(x) evaluado en phi: 3.236068
|g1'(phi)| = 3.236068


g2'(x) evaluado en phi: 0.309017
|g2'(phi)| = 0.309017
--------------------------------------------------
Análisis de convergencia:
-> g1(x) DIVERGIRÁ teóricamente puesto que |g1'(phi)| (3.24) > 1
-> g2(x) CONVERGIRÁ teóricamente puesto que |g2'(phi)| (0.31) < 1


**Ejercicio 6: Propagación de Errores a Mano**

Tienes dos aproximaciones con sus respectivas cotas de error absoluto:

* $\tilde{a} = 4.56$ con $|\Delta a| \le 0.14$ 
* $\tilde{b} = 1.23$ con $|\Delta b| \le 0.03$ 

1. Calcula la aproximación $\tilde{x}$ para la resta $x = a - b$.
2. Calcula la cota del error absoluto para esa resta ($|\Delta x|$).
3. Ahora supón que calculas el producto $x = a \cdot b$. Calcula las cotas de error relativo para $a$ y $b$ ($|\delta a|$ y $|\delta b|$), y úsalas para estimar la cota del error relativo del producto ($|\delta x|$).

In [6]:
# 1. Definición inicial de las cotas e incertidumbres
a = 4.56
delta_a = 0.14  # Error absoluto (Delta mayúscula)

b = 1.23
delta_b = 0.03

# 2. CÁLCULO DE LA RESTA: x = a - b
print("--- ANÁLISIS DEL ERROR PARA LA RESTA (SUMA O RESTA NORMAL) ---")
x_resta = a - b
print(f"Valor aproximado x_resta: {x_resta:.2f}")

# Cota del error absoluto en sumas y restas:
# Regla de oro -> "En sumas y restas, los errores absolutos (Deltas) SE SUMAN SIEMPRE".
delta_x_resta = delta_a + delta_b

# Resultado: x = (3.33 ± 0.17)
print(f"Cota error absoluto (resta): {delta_x_resta:.2f}")
print(f"Resultado final con cota: X = {x_resta:.2f} ± {delta_x_resta:.2f}\n")


# 3. CÁLCULO DEL PRODUCTO: x = a * b
print("\n--- ANÁLISIS DEL ERROR PARA EL PRODUCTO (MULTIPLICACIÓN) ---")
x_prod = a * b
print(f"Valor aproximado x_prod: {x_prod:.4f}")

# Regla de oro -> "En productos y divisiones, lo que SE SUMA SON LOS ERRORES RELATIVOS".
# Error relativo (delta minúscula) es el porcentaje de error: delta = Delta (abs) / |valor real estimado|
delta_rel_a_aprox = delta_a / abs(a)
delta_rel_b_aprox = delta_b / abs(b)

# Sumar errores relativos de a y b para estimar el total
delta_rel_x_prod = delta_rel_a_aprox + delta_rel_b_aprox

print(f"Cota error relativo de a (|delta a|): {delta_rel_a_aprox:.4f} ({delta_rel_a_aprox*100:.1f}%)")
print(f"Cota error relativo de b (|delta b|): {delta_rel_b_aprox:.4f} ({delta_rel_b_aprox*100:.1f}%)")
print(f"Cota error relativo total estimado (|delta x|): {delta_rel_x_prod:.4f} ({delta_rel_x_prod*100:.1f}%)")

# Para conseguir el error absoluto real de la multiplicación (por exigencias del examen o formato estandar),
# despejamos Delta = delta * |x|.
delta_abs_x_prod = delta_rel_x_prod * abs(x_prod)

# Resultado: x = (5.6088 ± 0.31)
print(f"Esa cota relativa pasada a valor absoluto (Delta x_prod): {delta_abs_x_prod:.2f}")
print(f"Resultado final con cota: X = {x_prod:.2f} ± {delta_abs_x_prod:.2f}")

# NOTA IMPORTANTE PARA EL EXAMEN:
# Como los valores de error originales tienen 1 o 2 cifras significativas máximas,
# la cota "0.3089" casi siempre se redondea hacia arriba en propagación pesimista a "0.31".


--- ANÁLISIS DEL ERROR PARA LA RESTA (SUMA O RESTA NORMAL) ---
Valor aproximado x_resta: 3.33
Cota error absoluto (resta): 0.17
Resultado final con cota: X = 3.33 ± 0.17


--- ANÁLISIS DEL ERROR PARA EL PRODUCTO (MULTIPLICACIÓN) ---
Valor aproximado x_prod: 5.6088
Cota error relativo de a (|delta a|): 0.0307 (3.1%)
Cota error relativo de b (|delta b|): 0.0244 (2.4%)
Cota error relativo total estimado (|delta x|): 0.0551 (5.5%)
Esa cota relativa pasada a valor absoluto (Delta x_prod): 0.31
Resultado final con cota: X = 5.61 ± 0.31
